# Senate Stage 2: heuristic blend of fundamentals, polling and expert ratings

`src/senate/stage2.py` blends three D-positive two-party margins with fixed weights. It is deterministic, with no
simulation:

| case | fundamentals | polling | experts |
|---|---|---|---|
| polling available | 0.37 | 0.54 | 0.09 |
| no polling | 0.65 | 0.00 | 0.35 |

- **Fundamentals:** Senate Stage 1 (`stage1.SENATE_S0` or `SENATE_S1`), out of sample. Each year is fit on earlier
  elections only.
- **Polling:** `polls.csv`'s `poll_margin_dem`. For 2016–2024 it's the 30-day RCP mean, ending the day before Election
  Day. For 2026 it's FiftyPlusOne's election-eve average. Missing polling selects the no-polling weights; it is
  never a 0 margin.
- **Experts:** Sabato 40%, Cook 30%, Inside Elections 30%, each rating as a margin (Solid ±31, Likely ±15, Lean ±10,
  Tilt ±5, Tossup 0).

The backtest covers 2016–2024, the years with ratings.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import senate.regression as sr
from senate import stage1, stage2

pd.set_option("display.width", 200)

In [2]:
STAGE1_CONFIGS = [stage1.SENATE_S0, stage1.SENATE_S1]
EVAL_YEARS = [2016, 2018, 2020, 2022, 2024]  # the years with expert ratings
print("weights:", stage2.WEIGHTS)
print("rating margins:", stage2.RATING_MARGIN, "providers:", stage2.PROVIDER_WEIGHTS)

data = sr.load_data()
polls, ratings = stage2.load_inputs()
blends = {}
for config in STAGE1_CONFIGS:
    rolling = stage1.rolling_predictions(config, eval_years=EVAL_YEARS, data=data)
    oos = pd.concat([r.predictions for r in rolling.values()], ignore_index=True)
    blends[config.name] = stage2.blend(oos, polls, ratings)
blends["senate_s1"].groupby("year")["weight_case"].value_counts().unstack()

weights: {'polling': {'fundamentals': 0.37, 'polling': 0.54, 'experts': 0.09}, 'no_polling': {'fundamentals': 0.65, 'polling': 0.0, 'experts': 0.35}}
rating margins: {'Solid D': 31.0, 'Likely D': 15.0, 'Lean D': 10.0, 'Tilt D': 5.0, 'Tossup': 0.0, 'Tilt R': -5.0, 'Lean R': -10.0, 'Likely R': -15.0, 'Solid R': -31.0} providers: {'sabato': 0.4, 'cook': 0.3, 'ie': 0.3}


weight_case,no_polling,polling
year,,
2016,10,22
2018,7,27
2020,14,20
2022,10,23
2024,16,17


## Backtest, pooled over 2016–2024

- `bias` = mean(actual − prediction): negative means the forecast was too Democratic.
- `ex_ante`: |partisan_lean + eve generic ballot| ≤ 10, known before the election.
- `bias_inc_*`: the bias for races with an R incumbent, an open seat, or a D incumbent.

In [3]:
COLUMNS = {"fundamentals only": "margin_fundamentals", "fundamentals + polling + experts": "prediction"}
table = pd.concat({name: stage2.score_table(b, data, COLUMNS) for name, b in blends.items()}, names=["stage1"])
table.round(2)

n   mae  bias  ex_ante_n  ex_ante_mae  accuracy_pct  ex_ante_accuracy_pct  bias_inc_R  bias_inc_open  bias_inc_D
stage1                                                                                                                                                          
senate_s0 fundamentals only                 166.0  6.59 -1.03       51.0         5.83         90.36                 74.51       -5.75          -1.21        3.30
          fundamentals + polling + experts  166.0  4.87 -1.83       51.0         3.82         92.77                 82.35       -4.24          -1.54        0.21
senate_s1 fundamentals only                 166.0  6.01 -1.11       51.0         4.98         92.17                 80.39       -3.92          -1.34        1.52
          fundamentals + polling + experts  166.0  4.76 -1.86       51.0         3.58         93.37                 84.31       -3.42          -1.66       -0.55

### Components on the polled races

Each component alone, on the races with a poll average (S1 as fundamentals). This shows what each one adds.

In [4]:
polled = blends["senate_s1"][blends["senate_s1"]["polling_available"]]
stage2.score_table(polled, data, {"S1": "margin_fundamentals", "polling": "margin_polling",
                                  "experts": "margin_experts", "blend": "prediction"}).round(2)

,n,mae,bias,ex_ante_n,ex_ante_mae,accuracy_pct,ex_ante_accuracy_pct,bias_inc_R,bias_inc_open,bias_inc_D
S1,109.0,6.12,-0.87,48.0,4.94,88.07,79.17,-3.72,-0.89,1.35
polling,109.0,4.80,-2.28,48.0,3.48,88.07,79.17,-3.86,-1.48,-1.42
experts,109.0,7.23,-4.03,48.0,6.70,92.59,85.11,-0.53,-2.66,-7.37
blend,109.0,4.35,-1.92,48.0,3.65,89.91,83.33,-3.51,-1.37,-0.93


### By year (MAE, all races)

In [5]:
by_year = {}
for name, b in blends.items():
    for label, column in COLUMNS.items():
        by_year[(name, label)] = (b["actual"] - b[column]).abs().groupby(b["year"]).mean()
pd.DataFrame(by_year).T.round(2)

year                                         2016  2018  2020  2022  2024
senate_s0 fundamentals only                 10.31  7.23  5.21  5.50  4.83
          fundamentals + polling + experts   7.44  3.24  5.18  4.52  4.08
senate_s1 fundamentals only                  9.13  6.39  4.98  5.17  4.49
          fundamentals + polling + experts   7.05  3.38  5.13  4.28  4.06

## 2026: the blended forecast (deterministic)

Fit on 2006–2024. Eleven races have FiftyPlusOne averages; the rest use the no-polling weights. NE, ID and SD have
no Democratic nominee (Osborn in NE is an independent), so their margins are the model's D-vs-R, or non-R-vs-R,
numbers.

In [6]:
forecast = {}
for config in STAGE1_CONFIGS:
    f = stage1.forecast(config, data=data)
    forecast[config.name] = stage2.blend(f.predictions, polls, ratings)
f26 = forecast["senate_s1"].set_index(["state", "class"])
f26 = f26.assign(prediction_with_s0=forecast["senate_s0"].set_index(["state", "class"])["prediction"])
cols = ["special", "margin_fundamentals", "margin_polling", "margin_experts", "weight_case", "prediction",
        "prediction_with_s0"]
f26[cols].sort_values("prediction").round(1)

,,special,margin_fundamentals,margin_polling,margin_experts,weight_case,prediction,prediction_with_s0
state,class,,,,,,,
WY,2,0,-35.8,NaN,-31.0,no_polling,-34.1,-35.6
WV,2,0,-34.0,NaN,-31.0,no_polling,-32.9,-32.6
ID,2,0,-28.4,NaN,-31.0,no_polling,-29.3,-28.7
OK,2,0,-24.5,NaN,-31.0,no_polling,-26.8,-27.8
AR,2,0,-23.1,NaN,-31.0,no_polling,-25.9,-25.1
SD,2,0,-21.7,NaN,-31.0,no_polling,-25.0,-24.1
TN,2,0,-21.1,NaN,-31.0,no_polling,-24.6,-23.7
KY,2,0,-19.9,NaN,-31.0,no_polling,-23.8,-24.6
AL,2,0,-19.7,NaN,-31.0,no_polling,-23.6,-24.4


In [7]:
close = f26[f26["prediction"].abs() <= 10].sort_values("prediction")
display(Markdown(f"**Within 10 points (S1 blend):** {len(close)} races"))
display(close[cols].round(1))
seats_d = {name: int((b["prediction"] > 0).sum()) for name, b in forecast.items()}
print("2026 races with a D-positive blended margin:", seats_d, "of", len(f26))

**Within 10 points (S1 blend):** 10 races

,,special,margin_fundamentals,margin_polling,margin_experts,weight_case,prediction,prediction_with_s0
state,class,,,,,,,
SC,2,0,-9.6,NaN,-8.5,no_polling,-9.2,-7.9
FL,3,1,-1.6,-6.5,-15.0,polling,-5.5,-5.5
IA,2,0,-2.9,0.6,-4.0,polling,-1.1,-1.2
AK,2,0,-6.1,2.4,4.0,polling,-0.6,0.2
KS,2,0,-9.2,4.6,4.0,polling,-0.5,0.2
TX,2,0,-2.6,3.9,5.5,polling,1.6,1.5
OH,3,1,-1.3,3.1,5.5,polling,1.7,1.6
MI,2,0,8.4,5.8,8.5,polling,7.0,7.1
ME,2,0,13.6,4.9,4.0,polling,8.0,9.3


2026 races with a D-positive blended margin: {'senate_s0': 19, 'senate_s1': 17} of 35
